# Improved CIC-IDS2017 Classical Baseline — Version 2

**Research phase:** Classical baseline development and quantum-comparison preparation  
**Task:** Binary network-intrusion classification — benign vs. malicious  
**Classical models:** Linear Support Vector Machine (SVM) and Random Forest  
**Initial split:** 70% training / 15% validation / 15% held-out test  
**Future quantum-comparison sample:** bounded sample selected only after the classical split

This notebook establishes reproducible classical reference models before later quantum or hybrid experiments.

> **Research integrity:** Do not describe results as completed until this notebook has actually been run on the improved dataset and the outputs have been reviewed.

## 1. Why use the improved CIC-IDS2017 release?

The original CICIDS2017 is widely used, but later peer-reviewed analyses identified problems involving flow construction, feature extraction, and ground-truth labeling. This notebook is designed for the corrected/improved release published by the DistriNet research group.

The notebook also records provenance and file hashes so the exact local data used in an experiment can be identified later.

In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, time, warnings
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = Path("../data/cicids2017_improved")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "Improved CIC-IDS2017"
DATASET_SOURCE = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/"
DATASET_DOCUMENTATION = "https://intrusion-detection.distrinet-research.be/CNS2022/CICIDS2017.html"

CLASSICAL_MAX_ROWS = 300_000  # Set None to use the full cleaned dataset.
TREAT_ATTEMPTED_AS_BENIGN = True
QUANTUM_TRAIN_N = 700
QUANTUM_VAL_N = 150
QUANTUM_TEST_N = 150
QUANTUM_FEATURE_COUNT = 8
RUN_SOURCE_FILE_SENSITIVITY = False
print("Configuration loaded.")

## 2. Dataset provenance and SHA-256 hashes

Place the extracted improved day-level CSV files under `data/cicids2017_improved/`. The raw dataset is not committed to GitHub. This cell computes SHA-256 hashes so the exact source files used can be documented.

In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}. Download and extract the improved CIC-IDS2017 dataset first.")

provenance = pd.DataFrame([{"file_name":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in csv_files])
display(provenance)
provenance.to_csv(RESULTS_DIR / "dataset_file_provenance.csv", index=False)

## 3. Load and inspect the improved dataset

The notebook preserves the source filename, an original row identifier, the original class label, and any attempted-flow indicator that is available.

In [ ]:
frames=[]
for path in csv_files:
    part=pd.read_csv(path, low_memory=False)
    part.columns=[str(c).strip() for c in part.columns]
    part["__source_file"]=path.name
    frames.append(part)
df=pd.concat(frames, ignore_index=True)
df["__row_id"]=np.arange(len(df), dtype=np.int64)
print(f"Loaded {len(csv_files)} files")
print("Raw shape:", df.shape)
print(df.columns.tolist())

## 4. Identify label and attempted-flow information

Improved releases can differ slightly in column naming. This cell uses normalized matching rather than assuming one exact spelling.

In [ ]:
def normalize_name(name):
    return "".join(ch.lower() for ch in str(name).strip() if ch.isalnum())
normalized={normalize_name(c):c for c in df.columns}
label_candidates=[normalized[k] for k in normalized if k in {"label","class","attacklabel"}]
if not label_candidates:
    label_candidates=[c for c in df.columns if "label" in normalize_name(c)]
if not label_candidates:
    raise KeyError("Could not identify a label column. Inspect df.columns and set LABEL_COL manually.")
LABEL_COL=label_candidates[0]
attempted_candidates=[c for c in df.columns if "attempt" in normalize_name(c)]
ATTEMPTED_COL=attempted_candidates[0] if attempted_candidates else None
print("Using label column:", LABEL_COL)
print("Attempted-flow column:", ATTEMPTED_COL)
df["__original_label"]=df[LABEL_COL].astype(str).str.strip()
display(df["__original_label"].value_counts().head(25))
if ATTEMPTED_COL is not None:
    display(df[ATTEMPTED_COL].value_counts(dropna=False).head(20))

## 5. Construct the binary target transparently

Initial target: benign = 0, malicious = 1.

The improved release distinguishes some flows as attempted attacks. The starter configuration treats attempted flows as benign for the initial binary task because the expected malicious behavior may not have completed. This choice is recorded and can be revisited later.

In [ ]:
label_upper=df["__original_label"].str.upper()
is_benign=label_upper.str.fullmatch(r"BENIGN|NORMAL", case=False, na=False)
is_attempted_label=label_upper.str.contains("ATTEMPT", na=False)
if ATTEMPTED_COL is not None:
    raw=df[ATTEMPTED_COL]
    if raw.dtype == bool:
        is_attempted_col=raw.fillna(False)
    else:
        is_attempted_col=raw.astype(str).str.strip().str.lower().isin({"1","true","yes","y","attempted"})
else:
    is_attempted_col=pd.Series(False, index=df.index)
is_attempted=is_attempted_label | is_attempted_col
df["target"]=(~is_benign).astype(int)
if TREAT_ATTEMPTED_AS_BENIGN:
    df.loc[is_attempted, "target"]=0
print(df["target"].value_counts())
print("Attempted flows detected:", int(is_attempted.sum()))
print("Treat attempted as benign:", TREAT_ATTEMPTED_AS_BENIGN)

## 6. Build a leakage-aware numeric feature matrix

Direct identifiers and metadata are excluded where present, including flow IDs, IP addresses, timestamps, source-file metadata, labels, and ports. Ports are conservatively excluded in this starter because they can encode scenario-specific shortcuts in benchmark datasets.

In [ ]:
DROP_NORMALIZED={"label","class","attacklabel","flowid","sourceip","srcip","destinationip","dstip","timestamp","datetime","sourceport","srcport","destinationport","dstport"}
drop_cols={LABEL_COL,"__source_file","__row_id","__original_label","target"}
if ATTEMPTED_COL is not None: drop_cols.add(ATTEMPTED_COL)
for c in df.columns:
    if normalize_name(c) in DROP_NORMALIZED: drop_cols.add(c)
feature_cols=[c for c in df.columns if c not in drop_cols]
X_all=df[feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
all_nan=X_all.columns[X_all.isna().all()].tolist()
if all_nan: X_all=X_all.drop(columns=all_nan)
y_all=df["target"].copy()
meta_all=df[["__row_id","__source_file","__original_label"]].copy()
print("Numeric feature matrix:", X_all.shape)
print("Excluded columns:", sorted(drop_cols))

## 7. Optional documented cap for the classical starter run

Set `CLASSICAL_MAX_ROWS = None` to use the full cleaned dataset. The default starter value applies a stratified cap to keep the first run practical on ordinary hardware. If used, the cap is recorded in metadata and should be disclosed.

In [ ]:
if CLASSICAL_MAX_ROWS is not None and len(X_all) > CLASSICAL_MAX_ROWS:
    selected_idx,_=train_test_split(np.arange(len(X_all)), train_size=CLASSICAL_MAX_ROWS, stratify=y_all, random_state=RANDOM_STATE)
    selected_idx=np.sort(selected_idx)
    X=X_all.iloc[selected_idx].reset_index(drop=True)
    y=y_all.iloc[selected_idx].reset_index(drop=True)
    meta=meta_all.iloc[selected_idx].reset_index(drop=True)
else:
    X=X_all.reset_index(drop=True); y=y_all.reset_index(drop=True); meta=meta_all.reset_index(drop=True)
print("Working classical dataset:", X.shape)
display(y.value_counts(normalize=True).rename("proportion"))

## 8. Create reproducible 70/15/15 classical partitions

The initial design uses 70% training, 15% validation, and 15% held-out test, stratified by the binary target. A later temporal or source-file sensitivity analysis should be performed before making strong generalization claims.

In [ ]:
idx=np.arange(len(X))
train_idx,temp_idx=train_test_split(idx,test_size=0.30,stratify=y,random_state=RANDOM_STATE)
val_idx,test_idx=train_test_split(temp_idx,test_size=0.50,stratify=y.iloc[temp_idx],random_state=RANDOM_STATE)
def take(indices): return X.iloc[indices].copy(), y.iloc[indices].copy(), meta.iloc[indices].copy()
X_train,y_train,meta_train=take(train_idx)
X_val,y_val,meta_val=take(val_idx)
X_test,y_test,meta_test=take(test_idx)
split_summary=pd.DataFrame({"rows":[len(X_train),len(X_val),len(X_test)],"fraction":[len(X_train)/len(X),len(X_val)/len(X),len(X_test)/len(X)],"malicious_rate":[y_train.mean(),y_val.mean(),y_test.mean()]}, index=["train","validation","test"])
display(split_summary)

## 9. Source-file distribution audit

Random stratification can place records from the same collection day in multiple partitions. This audit reports source-file representation so possible day/source artifacts are visible.

In [ ]:
def source_distribution(meta_part,name): return meta_part["__source_file"].value_counts(normalize=True).rename(name)
source_audit=pd.concat([source_distribution(meta_train,"train"),source_distribution(meta_val,"validation"),source_distribution(meta_test,"test")],axis=1).fillna(0)
display(source_audit)
source_audit.to_csv(RESULTS_DIR/"source_file_distribution.csv")

## 10. Classical model evaluation function

In [ ]:
def evaluate_model(name, model, X_eval, y_eval, train_seconds=None):
    start=time.perf_counter(); pred=model.predict(X_eval); infer=time.perf_counter()-start
    tn,fp,fn,tp=confusion_matrix(y_eval,pred,labels=[0,1]).ravel()
    fpr=fp/(fp+tn) if (fp+tn) else np.nan
    if hasattr(model,"decision_function"): score=model.decision_function(X_eval)
    elif hasattr(model,"predict_proba"): score=model.predict_proba(X_eval)[:,1]
    else: score=None
    auc=roc_auc_score(y_eval,score) if score is not None else np.nan
    return {"model":name,"accuracy":accuracy_score(y_eval,pred),"precision":precision_score(y_eval,pred,zero_division=0),"recall":recall_score(y_eval,pred,zero_division=0),"f1":f1_score(y_eval,pred,zero_division=0),"false_positive_rate":fpr,"roc_auc":auc,"train_seconds":train_seconds,"inference_seconds":infer,"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}

## 11. Large-sample classical baseline

These results are the larger classical reference. They are not yet the direct fair comparator for a future quantum kernel, because the quantum experiment will use a much smaller bounded sample.

In [ ]:
svm_model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("svm",LinearSVC(C=1.0,class_weight="balanced",random_state=RANDOM_STATE,max_iter=10000))])
start=time.perf_counter(); svm_model.fit(X_train,y_train); svm_train_seconds=time.perf_counter()-start
rf_model=Pipeline([("imputer",SimpleImputer(strategy="median")),("rf",RandomForestClassifier(n_estimators=300,class_weight="balanced_subsample",random_state=RANDOM_STATE,n_jobs=-1))])
start=time.perf_counter(); rf_model.fit(X_train,y_train); rf_train_seconds=time.perf_counter()-start
val_results=pd.DataFrame([evaluate_model("Linear SVM",svm_model,X_val,y_val,svm_train_seconds),evaluate_model("Random Forest",rf_model,X_val,y_val,rf_train_seconds)]).set_index("model")
display(val_results.round(4))

## 12. Held-out classical test evaluation

Use the held-out test set only after the baseline methodology is settled.

In [ ]:
test_results=pd.DataFrame([evaluate_model("Linear SVM",svm_model,X_test,y_test,svm_train_seconds),evaluate_model("Random Forest",rf_model,X_test,y_test,rf_train_seconds)]).set_index("model")
display(test_results.round(4))
test_results.to_csv(RESULTS_DIR/"classical_baseline_results.csv")

## 13. Confusion matrices

In [ ]:
for model_name,model in [("Linear SVM",svm_model),("Random Forest",rf_model)]:
    pred=model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=["Benign","Malicious"],values_format="d")
    plt.title(f"{model_name} — Held-Out Test Set")
    plt.show()

## 14. Prepare a bounded future quantum-comparison subset

The full dataset is **not** intended to be sent into a quantum-kernel simulator. Instead, a bounded sample is selected separately within the already-created train, validation, and held-out test partitions. The default starter target is 700/150/150. The exact number may be adjusted after pilot measurements of memory use and execution time.

In [ ]:
def stratified_bounded_sample(Xp,yp,mp,n,seed):
    n=min(n,len(Xp))
    if n==len(Xp): chosen=np.arange(len(Xp))
    else: chosen,_=train_test_split(np.arange(len(Xp)),train_size=n,stratify=yp,random_state=seed)
    chosen=np.sort(chosen)
    return Xp.iloc[chosen].reset_index(drop=True), yp.iloc[chosen].reset_index(drop=True), mp.iloc[chosen].reset_index(drop=True)
X_qtrain,y_qtrain,meta_qtrain=stratified_bounded_sample(X_train.reset_index(drop=True),y_train.reset_index(drop=True),meta_train.reset_index(drop=True),QUANTUM_TRAIN_N,RANDOM_STATE+1)
X_qval,y_qval,meta_qval=stratified_bounded_sample(X_val.reset_index(drop=True),y_val.reset_index(drop=True),meta_val.reset_index(drop=True),QUANTUM_VAL_N,RANDOM_STATE+2)
X_qtest,y_qtest,meta_qtest=stratified_bounded_sample(X_test.reset_index(drop=True),y_test.reset_index(drop=True),meta_test.reset_index(drop=True),QUANTUM_TEST_N,RANDOM_STATE+3)
quantum_subset_summary=pd.DataFrame({"rows":[len(X_qtrain),len(X_qval),len(X_qtest)],"malicious_rate":[y_qtrain.mean(),y_qval.mean(),y_qtest.mean()]},index=["train","validation","test"])
display(quantum_subset_summary)

## 15. Save a quantum-subset manifest without raw features

The manifest contains only original row identifier, source file, original label, binary target, and partition. It does not redistribute the raw feature rows.

In [ ]:
def manifest(mp,yp,partition):
    out=mp.copy(); out["target"]=yp.to_numpy(); out["partition"]=partition; return out
q_manifest=pd.concat([manifest(meta_qtrain,y_qtrain,"train"),manifest(meta_qval,y_qval,"validation"),manifest(meta_qtest,y_qtest,"test")],ignore_index=True)
display(q_manifest.head())
q_manifest.to_csv(RESULTS_DIR/"quantum_subset_manifest.csv",index=False)

## 16. Select a reduced feature set using training data only

A future quantum-kernel experiment should use a small feature representation. This starter uses mutual information as one transparent feature-ranking method. Ranking is performed on the bounded quantum training subset only so validation/test data do not influence feature selection.

In [ ]:
q_imputer=SimpleImputer(strategy="median")
X_qtrain_imp=q_imputer.fit_transform(X_qtrain)
mi=mutual_info_classif(X_qtrain_imp,y_qtrain,random_state=RANDOM_STATE)
mi_table=pd.DataFrame({"feature":X_qtrain.columns,"mutual_information":mi}).sort_values("mutual_information",ascending=False)
selected_features=mi_table.head(min(QUANTUM_FEATURE_COUNT,len(mi_table)))["feature"].tolist()
display(mi_table.head(20))
print("Selected features:", selected_features)
with open(RESULTS_DIR/"quantum_candidate_features.json","w",encoding="utf-8") as f:
    json.dump({"method":"mutual_info_classif","fit_partition":"bounded quantum-comparison training subset only","feature_count":len(selected_features),"features":selected_features},f,indent=2)

## 17. Same-sample, same-feature classical comparator

This is the classical result that should eventually be compared directly with a quantum-kernel model. It uses the same bounded observations, reduced feature set, and train/validation/test boundaries.

In [ ]:
Xq_tr=X_qtrain[selected_features]; Xq_te=X_qtest[selected_features]
svm_qcomp=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("svm",LinearSVC(C=1.0,class_weight="balanced",random_state=RANDOM_STATE,max_iter=10000))])
start=time.perf_counter(); svm_qcomp.fit(Xq_tr,y_qtrain); svm_q_train_seconds=time.perf_counter()-start
rf_qcomp=Pipeline([("imputer",SimpleImputer(strategy="median")),("rf",RandomForestClassifier(n_estimators=300,class_weight="balanced_subsample",random_state=RANDOM_STATE,n_jobs=-1))])
start=time.perf_counter(); rf_qcomp.fit(Xq_tr,y_qtrain); rf_q_train_seconds=time.perf_counter()-start
qcomp_results=pd.DataFrame([evaluate_model("Linear SVM — bounded/reduced",svm_qcomp,Xq_te,y_qtest,svm_q_train_seconds),evaluate_model("Random Forest — bounded/reduced",rf_qcomp,Xq_te,y_qtest,rf_q_train_seconds)]).set_index("model")
display(qcomp_results.round(4))
qcomp_results.to_csv(RESULTS_DIR/"bounded_reduced_classical_comparator.csv")

## 18. Optional source-file/day holdout sensitivity scaffold

A random 70/15/15 split is useful for a reproducible baseline, but it does not fully test whether the models generalize across different collection days. This optional scaffold separates entire source files. Because there are few source files, exact proportions may differ from 70/15/15.

In [ ]:
if RUN_SOURCE_FILE_SENSITIVITY:
    groups=meta["__source_file"].to_numpy(); idx_all=np.arange(len(X))
    gss1=GroupShuffleSplit(n_splits=1,train_size=0.70,random_state=RANDOM_STATE)
    gtrain_idx,grem_idx=next(gss1.split(idx_all,y,groups=groups))
    rem_groups=groups[grem_idx]
    gss2=GroupShuffleSplit(n_splits=1,train_size=0.50,random_state=RANDOM_STATE+1)
    gval_rel,gtest_rel=next(gss2.split(grem_idx,y.iloc[grem_idx],groups=rem_groups))
    gval_idx=grem_idx[gval_rel]; gtest_idx=grem_idx[gtest_rel]
    print("Train files:",sorted(meta.iloc[gtrain_idx]["__source_file"].unique()))
    print("Validation files:",sorted(meta.iloc[gval_idx]["__source_file"].unique()))
    print("Test files:",sorted(meta.iloc[gtest_idx]["__source_file"].unique()))
else:
    print("Source-file sensitivity analysis is disabled by default.")

## 19. Save run metadata

The metadata file provides a machine-readable record of the experiment configuration. Review it before committing generated results.

In [ ]:
run_metadata={"dataset_name":DATASET_NAME,"dataset_source":DATASET_SOURCE,"dataset_documentation":DATASET_DOCUMENTATION,"random_state":RANDOM_STATE,"classical_max_rows":CLASSICAL_MAX_ROWS,"treat_attempted_as_benign":TREAT_ATTEMPTED_AS_BENIGN,"raw_loaded_rows":int(len(df)),"working_rows":int(len(X)),"working_feature_count":int(X.shape[1]),"train_rows":int(len(X_train)),"validation_rows":int(len(X_val)),"test_rows":int(len(X_test)),"quantum_train_target":QUANTUM_TRAIN_N,"quantum_validation_target":QUANTUM_VAL_N,"quantum_test_target":QUANTUM_TEST_N,"quantum_feature_count":int(len(selected_features)),"selected_quantum_candidate_features":selected_features,"software":{"python":sys.version,"platform":platform.platform(),"pandas":pd.__version__,"numpy":np.__version__,"scikit_learn":sklearn.__version__,"matplotlib":matplotlib.__version__}}
with open(RESULTS_DIR/"run_metadata.json","w",encoding="utf-8") as f: json.dump(run_metadata,f,indent=2)
print(json.dumps(run_metadata,indent=2))

## 20. Interpretation template

Complete this section only after the notebook runs successfully.

### Dataset provenance
- Improved CIC-IDS2017 release/source:
- Files used:
- SHA-256 manifest reviewed:
- Treatment of attempted flows:

### Classical baseline
- Full/working rows:
- Sampling cap used, if any:
- Feature count:
- Train/validation/test sizes:
- Linear SVM metrics:
- Random Forest metrics:

### Bounded quantum-comparison preparation
- Bounded train/validation/test sizes:
- Reduced feature count:
- Selected feature names:
- Same-sample SVM metrics:
- Same-sample Random Forest metrics:

### Leakage and robustness notes
- Source-file distribution across random partitions:
- Any obvious source/day imbalance:
- Whether source-file/day holdout sensitivity analysis was run:
- Any concerns about temporal proximity or benchmark artifacts:

### Required limitations
Discuss at minimum:
- corrected/improved CIC-IDS2017 is still a benchmark rather than live enterprise telemetry;
- binary relabeling simplifies the original multiclass problem;
- a stratified random split may not prove temporal generalization;
- a bounded quantum sample trades representativeness for computational feasibility;
- simulator feasibility does not equal physical quantum-hardware feasibility;
- none of these results demonstrate live real-time quantum threat detection.